<a href="https://colab.research.google.com/github/faizanarif2/PyTorch_NeuralNetworks/blob/main/CNN_SyntheticLineDataset.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision.datasets import MNIST
from torch.utils.data import DataLoader,Dataset
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint,EarlyStopping

In [6]:
class SyntheticLineDataset(Dataset):
#Made custom Dataset usinng MNIST
#Made synthetic line images
    def __init__(self, base_dataset, seq_len=4):
        self.base_dataset = base_dataset
        self.seq_len = seq_len

    def __len__(self):
        return len(self.base_dataset) // self.seq_len

    def __getitem__(self, idx):
        images, labels = [], []
        for i in range(self.seq_len):
            img, label = self.base_dataset[idx * self.seq_len + i]
            images.append(img)
            labels.append(label)

        line_image = torch.cat(images, dim=2)
        line_labels = torch.tensor(labels, dtype=torch.long)
        return line_image, line_labels

In [ ]:
class DataModule(pl.LightningDataModule):

  def __init__(self,data_dir="./data",batch_size: int=32,seq_len: int=4):

    super().__init__()
    self.data_dir=data_dir
    self.batch_size=batch_size
    self.seq_len=seq_len
    self.transform=transforms.ToTensor()

  def prepare_data(self):
    MNIST(self.data_dir,train=True,download=True)
    MNIST(self.data_dir,train=False,download=True)

  def setup(self,stage=None):
    raw_train=MNIST(self.data_dir,train=True,transform=self.transform)
    raw_val=MNIST(self.data_dir,train=False,transform=self.transform)

    self.train_dataset=SyntheticLineDataset(raw_train,seq_len=self.seq_len)
    self.val_dataset=SyntheticLineDataset(raw_val,seq_len=self.seq_len)

  def train_dataloader(self):
    return DataLoader(self.train_dataset,batch_size=self.batch_size,shuffle=True)

  def val_dataloader(self):
    return DataLoader(self.val_dataset,batch_size=self.batch_size)




